In [3]:
import duckdb
from pathlib import Path

con = duckdb.connect()
path = (Path("..") / "data" / "processed" / "train_supervised.csv").as_posix()

q1 = f"""
WITH g AS (
  SELECT body_norm, recommendation_status AS y, COUNT(*) AS n
  FROM read_csv('{path}', header=true, all_varchar=true)
  WHERE recommendation_status IS NOT NULL AND body_norm IS NOT NULL
  GROUP BY body_norm, recommendation_status
), t AS (
  SELECT body_norm, SUM(n) AS total, MAX(n) AS majority
  FROM g GROUP BY body_norm HAVING SUM(n) >= 5
)
SELECT COUNT(*) AS distinct_texts, SUM(total) AS rows_covered,
       SUM(majority) AS rows_with_majority_label,
       ROUND(100.0 * (1 - SUM(majority) / SUM(total)), 2) AS pct_rows_disagreeing_with_majority
FROM t
"""
print(con.sql(q1).df())

q2 = f"""
SELECT body_norm, recommendation_status, COUNT(*) AS n,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY body_norm), 1) AS pct_of_this_text
FROM read_csv('{path}', header=true, all_varchar=true)
WHERE body_norm IN ('عالی', 'عالیه', 'خوب', 'بد', 'افتضاح', 'نخرید')
  AND recommendation_status IS NOT NULL
GROUP BY body_norm, recommendation_status
ORDER BY body_norm, n DESC
"""
print(con.sql(q2).df())

   distinct_texts  rows_covered  rows_with_majority_label  \
0           23323      842251.0                  780979.0   

   pct_rows_disagreeing_with_majority  
0                                7.27  
   body_norm recommendation_status      n  pct_of_this_text
0     افتضاح       not_recommended    570              92.8
1     افتضاح               no_idea     37               6.0
2     افتضاح           recommended      7               1.1
3         بد       not_recommended     52              66.7
4         بد               no_idea     16              20.5
5         بد           recommended     10              12.8
6        خوب           recommended  25323              92.8
7        خوب               no_idea   1839               6.7
8        خوب       not_recommended    131               0.5
9       عالی           recommended  53289              99.0
10      عالی               no_idea    348               0.6
11      عالی       not_recommended    169               0.3
12     عالیه     